# mu-logsigma-encoder-head — faded example 2: Copy single-head weights into a two-head encoder by row blocks

> Practice drill from [Delta Drills](https://delta-drills.vercel.app). Atom: `mu-logsigma-encoder-head`. The last cell reports your progress on the `VAE: mu+logsigma encoder head` subtopic back to Delta Drills.

**Most of the code is already written — complete the one blanked step**, run the test to check it, then run the last cell to record your progress.

## Setup

In [ ]:
import numpy as np
import torch as t
from torch import Tensor
import einops
from einops import rearrange, reduce, repeat

t.manual_seed(0)
np.random.seed(0)
import matplotlib.pyplot as plt

## Connect to Delta Drills

Paste your Delta Drills auth token below so this drill can report progress on the `VAE: mu+logsigma encoder head` subtopic. Copy it from your Delta Drills account page.

This standalone exercises the atom **`mu-logsigma-encoder-head`** (exercise 1). Completion fires the beacon at the bottom.

In [ ]:
# === Delta Drills auth ===
DD_TOKEN = ""  # paste your token here, then run this cell
DD_ATOM_ID = "mu-logsigma-encoder-head"
DD_SUBTOPIC = "VAE: mu+logsigma encoder head"
DD_BACKEND_URL = "https://delta-drills-backend.fly.dev"

_dd_passed = set()

## Concept

_First time on this topic? Run the **Setup** cell above and skim it: every class and helper mentioned below is defined there. You don't need to have done any other drill first._

A single `Linear(d_in, 2*latent)` and two separate `Linear(d_in, latent)` heads are mathematically equivalent when the two-head weights are the top and bottom row-blocks of the single-head weight matrix. Copying weights in-place with `.data.copy_()` establishes this equivalence without touching autograd.

## Faded exercise 2

Implement `copy_to_two_head(fc_mu, fc_logsigma, single_linear)` that:
1. Copies row block `[0:latent_dim]` of `single_linear.weight` into `fc_mu.weight.data`.
2. Copies row block `[latent_dim:]` of `single_linear.weight` into `fc_logsigma.weight.data`.
3. Does the same splits for the bias vectors.

Your task: **fill in the four `.data.copy_()` calls** (weight and bias for each head).

**Your task:** complete the one blanked step in the code cell below. The surrounding code, function signatures, and variable names are given — work out the missing expression yourself, then run the test.

In [ ]:
import torch
import torch.nn as nn

def copy_to_two_head(fc_mu: nn.Linear, fc_logsigma: nn.Linear, single_linear: nn.Linear):
    latent_dim = fc_mu.out_features
    fc_mu.weight.data.copy_(single_linear.weight.data[:latent_dim])
    fc_mu.bias.data.copy_(single_linear.bias.data[:latent_dim])
    fc_logsigma.weight.data.copy_(single_linear.weight.data[latent_dim:])
    fc_logsigma.bias.data.copy_(single_linear.bias.data[latent_dim:])

def _test():
    import torch, torch.nn as nn
    torch.manual_seed(0)
    d_in, L = 16, 6
    single = nn.Linear(d_in, 2 * L)
    fc_mu  = nn.Linear(d_in, L)
    fc_ls  = nn.Linear(d_in, L)
    copy_to_two_head(fc_mu, fc_ls, single)
    h = torch.randn(4, d_in)
    mu_2h  = fc_mu(h)
    ls_2h  = fc_ls(h)
    mu_1h, ls_1h = single(h).chunk(2, dim=-1)
    assert torch.allclose(mu_2h,  mu_1h,  atol=1e-6)
    assert torch.allclose(ls_2h, ls_1h, atol=1e-6)


def _test():
    import torch, torch.nn as nn
    torch.manual_seed(0)
    d_in, L = 16, 6
    single = nn.Linear(d_in, 2 * L)
    fc_mu  = nn.Linear(d_in, L)
    fc_ls  = nn.Linear(d_in, L)
    copy_to_two_head(fc_mu, fc_ls, single)
    torch.manual_seed(3)
    h = torch.randn(4, d_in)
    # Two-head forward
    mu_2h = fc_mu(h)
    ls_2h = fc_ls(h)
    # Single-head forward split
    mu_1h, ls_1h = single(h).chunk(2, dim=-1)
    assert torch.allclose(mu_2h, mu_1h, atol=1e-6), f"mu mismatch: {(mu_2h - mu_1h).abs().max()}"
    assert torch.allclose(ls_2h, ls_1h, atol=1e-6), f"ls mismatch"
    # Verify weights were actually copied, not just coincidentally equal
    assert torch.allclose(fc_mu.weight.data, single.weight.data[:L])
    assert torch.allclose(fc_ls.weight.data, single.weight.data[L:])


try:
    _test()
    _dd_passed.add('faded2')
    print('[Delta Drills] faded2 passed.')
except AssertionError as _e:
    print('Test failed:', _e)

## Report your progress

Run the cell below to send your progress to Delta Drills. It only counts if the test above passed.

In [ ]:
# === Delta Drills completion beacon ===
import urllib.request as _dd_req, json as _dd_json

_DD_REQUIRED = {'faded2'}

def report_completion():
    missing = _DD_REQUIRED - _dd_passed
    if missing:
        print(f"[Delta Drills] {sorted(missing)} not yet passing — fix the cell above, then re-run this one.")
        return
    if not DD_TOKEN:
        print('[Delta Drills] DD_TOKEN is empty — completion not reported.')
        return
    body = _dd_json.dumps({
        'exercise_title': f'procedural-drill:{DD_ATOM_ID}:faded1',
        'subtopics': [DD_SUBTOPIC],
        'feedback': 'somewhat',
        'correct': True,
    }).encode('utf-8')
    req = _dd_req.Request(
        f'{DD_BACKEND_URL}/api/practice/arena-rating',
        data=body,
        headers={
            'Content-Type': 'application/json',
            'Authorization': f'Bearer {DD_TOKEN}',
        },
        method='POST',
    )
    try:
        with _dd_req.urlopen(req, timeout=5) as r:
            resp = _dd_json.loads(r.read())
        print(f'[Delta Drills] reported {DD_ATOM_ID} (subtopic={DD_SUBTOPIC!r})')
        print(f'[Delta Drills] EWMA updated: {resp}')
    except Exception as e:
        print(f'[Delta Drills] beacon failed: {e}')

report_completion()

<details><summary>Solution</summary>

```python
import torch
import torch.nn as nn

def copy_to_two_head(fc_mu: nn.Linear, fc_logsigma: nn.Linear, single_linear: nn.Linear):
    latent_dim = fc_mu.out_features
    fc_mu.weight.data.copy_(single_linear.weight.data[:latent_dim])
    fc_mu.bias.data.copy_(single_linear.bias.data[:latent_dim])
    fc_logsigma.weight.data.copy_(single_linear.weight.data[latent_dim:])
    fc_logsigma.bias.data.copy_(single_linear.bias.data[latent_dim:])

def _test():
    import torch, torch.nn as nn
    torch.manual_seed(0)
    d_in, L = 16, 6
    single = nn.Linear(d_in, 2 * L)
    fc_mu  = nn.Linear(d_in, L)
    fc_ls  = nn.Linear(d_in, L)
    copy_to_two_head(fc_mu, fc_ls, single)
    h = torch.randn(4, d_in)
    mu_2h  = fc_mu(h)
    ls_2h  = fc_ls(h)
    mu_1h, ls_1h = single(h).chunk(2, dim=-1)
    assert torch.allclose(mu_2h,  mu_1h,  atol=1e-6)
    assert torch.allclose(ls_2h, ls_1h, atol=1e-6)
```
</details>